In [ ]:
import rasterio
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from pathlib import Path
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.patches import Patch, FancyArrow, Polygon


In [ ]:
def add_north_arrow(ax, x=0.95, y=0.05, arrow_length=0.08, width=0.03):
    """
    Add a north arrow to the map.

    Args:
        ax (matplotlib.axes.Axes): Axis to draw the arrow on.
        x (float): Horizontal position in axes coordinates (0-1). Defaults to 0.95.
        y (float): Vertical position in axes coordinates (0-1). Defaults to 0.05.
        arrow_length (float): Length of the arrow in axes coordinates. Defaults
            to 0.08.
        width (float): Width of the arrow base in axes coordinates. Defaults to
            0.03.

    Returns:
        None
    """
    # Arrow coordinates in axes coordinates
    arrow_x = x
    arrow_y = y
    
    # Create arrow shaft (rectangle)
    shaft_width = width * 0.3
    shaft = plt.Rectangle((arrow_x - shaft_width/2, arrow_y), 
                          shaft_width, arrow_length * 0.6,
                          transform=ax.transAxes, 
                          facecolor='black', 
                          edgecolor='black',
                          zorder=1000)
    
    # Create arrow head (triangle)
    head_height = arrow_length * 0.4
    head_width = width
    triangle = np.array([
        [arrow_x, arrow_y + arrow_length * 0.6 + head_height],  # tip
        [arrow_x - head_width/2, arrow_y + arrow_length * 0.6],  # bottom left
        [arrow_x + head_width/2, arrow_y + arrow_length * 0.6]   # bottom right
    ])
    
    arrowhead = Polygon(triangle, 
                       transform=ax.transAxes,
                       facecolor='black',
                       edgecolor='black',
                       zorder=1000)
    
    # Add 'N' label
    ax.text(arrow_x, arrow_y - 0.02, 'N', 
           transform=ax.transAxes,
           ha='center', va='top',
           fontsize=20, fontweight='bold',
           zorder=1000, color="black")
    
    ax.add_patch(shaft)
    ax.add_patch(arrowhead)

    return None
    

## File I/O

In [ ]:
# Define a path to the project folder
project_folder = "<PATH/TO/PROJECT/FOLDER>"

# Define a path to the hillshade layer
hillshade_path = f"{project_folder}/data/rasters/Western_US_Hillshade_250m.tif"

# Specify your folder path here
shapefile_folder = f"{project_folder}/data/access_class_gpkgs/"
western_states_path = f"{project_folder}/data/shapefiles/westernstates.shp"

# Define the output path
output_path1 = f"{project_folder}/figures/nfs_visualization.png"
output_path2 = f"{project_folder}/figures/nfs_visualization_inset.png"


## Format visualization layers

In [ ]:
# Read in the the shapefiles
western_states = gpd.read_file(western_states_path)
roaded = gpd.read_file(Path(shapefile_folder) / "roaded_nfs.gpkg")
roadless = gpd.read_file(Path(shapefile_folder) / "roadless_nfs.gpkg")
wilderness = gpd.read_file(Path(shapefile_folder) / "wilderness_nfs.gpkg")

# Reproject to Web Mercator (EPSG:3857) for basemap compatibility
print("\nReprojecting to Web Mercator...")
western_states = western_states.to_crs(epsg=3857)
roaded = roaded.to_crs(epsg=3857)
roadless = roadless.to_crs(epsg=3857)
wilderness = wilderness.to_crs(epsg=3857)

# Read and reproject hillshade raster
print("Reading hillshade raster...")
with rasterio.open(hillshade_path) as src:
    if src.crs.to_epsg() != 3857:
        dst_transform, dst_width, dst_height = calculate_default_transform(
            src.crs, 'EPSG:3857', src.width, src.height, *src.bounds
        )
        hillshade = np.empty((dst_height, dst_width), dtype=src.dtypes[0])
        reproject(
            source=rasterio.band(src, 1),
            destination=hillshade,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs='EPSG:3857',
            resampling=Resampling.bilinear
        )
    else:
        hillshade = src.read(1)
        dst_transform = src.transform
        dst_height, dst_width = hillshade.shape

# Compute imshow extent [left, right, bottom, top]
hs_extent = [
    dst_transform[2],
    dst_transform[2] + dst_transform[0] * dst_width,
    dst_transform[5] + dst_transform[4] * dst_height,
    dst_transform[5]
]

# Mask nodata (0) and apply linear stretch 75–255 -> 0–1
hillshade = hillshade.astype(np.float32)
hillshade[hillshade == 0] = np.nan
hillshade = np.clip(hillshade, 50, 255)
hillshade = ((hillshade - 50.0) / (255.0 - 50.0)) + 0.25



## Generate the figure

In [ ]:
# Create figure
print("\nCreating visualization...")
fig, ax = plt.subplots(figsize=(10, 12))

# Set extent FIRST so basemap knows what area to fetch
minx, miny, maxx, maxy = western_states.total_bounds
pad = 100_000
ax.set_xlim(minx - pad, maxx + pad)
ax.set_ylim(miny - pad, maxy + pad)

# ESRI Ocean basemap (lowest layer)
cx.add_basemap(ax, source=cx.providers.Esri.WorldPhysical, attribution=False, zorder=0)

# Hillshade on top of basemap
print("Rendering hillshade basemap...")
ax.imshow(hillshade, extent=hs_extent, cmap='gray', vmin=-0.1, vmax=1,
          aspect='auto', interpolation='bilinear', zorder=1)

# Vector layers on top of hillshade
western_states.plot(ax=ax, color='none', edgecolor='#333333', linewidth=1.5, alpha=1.0, zorder=4)
roaded.plot(ax=ax, color="#EE9A00", alpha=0.7, linewidth=0, zorder=2)
roadless.plot(ax=ax, color="#6E8B3D", alpha=0.7, linewidth=0, zorder=2)
wilderness.plot(ax=ax, color="#1874CD", alpha=0.7, linewidth=0, zorder=2)

ax.set_aspect('equal')

# Legend
legend_elements = [
    Patch(facecolor="#EE9A00", alpha=0.7, label='Developed'),
    Patch(facecolor="#6E8B3D", alpha=0.7, label='IRA'),
    Patch(facecolor="#1874CD", alpha=0.7, label='Wilderness')
]
legend = ax.legend(handles=legend_elements, loc='lower left', fontsize=15, framealpha=0.9,
                   edgecolor='black', bbox_to_anchor=(0.015, 0.05), title=None)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')

scalebar = ScaleBar(1, units='m', location='lower left', length_fraction=0.2,
                    fixed_value=750, fixed_units='km', box_alpha=0,
                    color='black', font_properties={'size': 16, 'weight': 'bold'},
                    pad=0.5, sep=5, scale_loc='bottom')
ax.add_artist(scalebar)

add_north_arrow(ax, x=0.96, y=0.15, arrow_length=0.08, width=0.03)

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(2)

plt.tight_layout()
plt.savefig(output_path1, dpi=300, bbox_inches='tight')
print(f"\nVisualization saved to: {output_path1}")
plt.show()
